# Модель SIR в виде сети Петри: набор параметров

При $\beta$ от 0.1 до 0.8 заражается всё население сразу, потому что
скорость заражения в модели равна $\beta S I$ и при $S \approx 1000$
получается очень большой. Чтобы увидеть порог эпидемии, возьмём
маленькие значения $\beta$ и несколько значений $\gamma$.
Порог определяется числом $R_0 = \beta N / \gamma$, где $N = 1000$:
эпидемия развивается только при $R_0 > 1$.

## Активация проекта и загрузка пакетов

In [1]:
using DrWatson
@quickactivate "project"
@isdefined(SIRPetri) || include(srcdir("SIRPetri.jl"))
using .SIRPetri
using DataFrames, CSV, Plots

script_name = "sirpetri_params"
mkpath(plotsdir())
mkpath(datadir())

"/home/iaberezhnoy/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab06/project/data"

## Набор параметров

Параметры собраны в словарь. Функция `dict_list` строит все
комбинации: 6 значений $\beta$ и 3 значения $\gamma$ --- всего 18 запусков.

In [2]:
param_dict = Dict(
    :β => [0.00005, 0.0001, 0.0002, 0.0003, 0.0005, 0.001],
    :γ => [0.05, 0.1, 0.2],
    :tmax => 300.0,
)

params_list = dict_list(param_dict)
length(params_list)

18

## Функция одного эксперимента

Функция запускает детерминированную симуляцию и возвращает
число $R_0$, пик числа инфицированных, время пика
и конечное число выздоровевших.

In [3]:
function run_experiment(p)
    β = p[:β]
    γ = p[:γ]
    net, u0, _ = build_sir_network(β, γ)
    df = simulate_deterministic(net, u0, (0.0, p[:tmax]), saveat = 0.1, rates = [β, γ])
    k = argmax(df.I)
    return (
        β = β,
        γ = γ,
        R0 = round(β * sum(u0) / γ, digits = 2),
        peak_I = round(df.I[k], digits = 1),
        t_peak = df.time[k],
        final_R = round(df.R[end], digits = 1),
    )
end

run_experiment (generic function with 1 method)

## Запуск экспериментов

In [4]:
results = []
for p in params_list
    push!(results, run_experiment(p))
end

df_params = DataFrame(results)
sort!(df_params, [:γ, :β])
CSV.write(datadir("sir_params.csv"), df_params)
println(df_params)

18×6 DataFrame
 Row │ β        γ        R0       peak_I   t_peak   final_R 
     │ Float64  Float64  Float64  Float64  Float64  Float64 
─────┼──────────────────────────────────────────────────────
   1 │  5.0e-5     0.05     1.0      10.0      0.0    106.1
   2 │  0.0001     0.05     2.0     158.5     87.6    799.0
   3 │  0.0002     0.05     4.0     405.9     38.8    980.4
   4 │  0.0003     0.05     6.0     536.4     25.5    997.5
   5 │  0.0005     0.05    10.0     670.8     15.5   1000.0
   6 │  0.001      0.05    20.0     800.8      8.1   1000.0
   7 │  5.0e-5     0.1      0.5      10.0      0.0     19.7
   8 │  0.0001     0.1      1.0      10.0      0.0    131.0
   9 │  0.0002     0.1      2.0     158.5     43.8    800.2
  10 │  0.0003     0.1      3.0     303.8     26.6    941.2
  11 │  0.0005     0.1      5.0     480.1     15.3    993.1
  12 │  0.001      0.1     10.0     670.8      7.8   1000.0
  13 │  5.0e-5     0.2      0.25     10.0      0.0     13.3
  14 │  0.0001     0.2

## Графики

Слева --- пик числа инфицированных, справа --- конечное число
выздоровевших в зависимости от $\beta$ для каждого значения $\gamma$.

In [ ]:
p1 = plot(xlabel = "β", ylabel = "Peak I", title = "Пик эпидемии", legend = :topleft)
p2 = plot(xlabel = "β", ylabel = "Final R", title = "Переболело", legend = :bottomright)
for γ in sort(unique(df_params.γ))
    sub = df_params[df_params.γ .== γ, :]
    plot!(p1, sub.β, sub.peak_I, marker = :circle, linewidth = 2, label = "γ = $γ")
    plot!(p2, sub.β, sub.final_R, marker = :circle, linewidth = 2, label = "γ = $γ")
end
plt = plot(p1, p2, layout = (1, 2), size = (900, 400), left_margin = 5Plots.mm, bottom_margin = 5Plots.mm)

## Сохранение графика

In [ ]:
savefig(plt, plotsdir("sir_params.png"))
println("Результаты сохранены в data/sir_params.csv и plots/sir_params.png")